# Figure 2B: inter-method correlation of delta-22 ¹H stationary shieldings

Lower-triangle Pearson correlation matrix of per-site ¹H delta-22 stationary shieldings across methods
(pcSseg-2 basis). The colorbar is Pearson r; the published panel mislabels it r², a typo, corrected here.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np

import delta22
import paths
import fig2b_plots

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# Gas-phase ("stationary") shieldings, one value per (solute, geometry, basis, site, nucleus, method).
# The stationary shielding does not depend on solvent, so keep one solvent (TIP4P) to deduplicate.
NUCLEUS = "H"
BASIS = "pcSseg2"

dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
corr_df = dft[["solute", "sap_geometry_type", "sap_nmr_method", "sap_basis",
               "nucleus", "site", "solvent", "stationary"]].copy()
corr_df.columns = [c.replace("sap_", "") for c in corr_df.columns]
corr_df = corr_df.query('solvent == "TIP4P"').drop(columns=["solvent"])

pivot = corr_df.pivot(index=["solute", "geometry_type", "basis", "site", "nucleus"],
                      columns="nmr_method", values="stationary")
sub = pivot.query("nucleus == @NUCLEUS and basis == @BASIS")
correlations = sub.corr()
off_diag = correlations.where(~np.eye(len(correlations), dtype=bool))
print(f"{correlations.shape[0]} methods; worst pairwise Pearson r = {off_diag.min().min():.5f}")

In [ ]:
fig2b_plots.plot_correlation_matrix(correlations, save_png=figure_path("fig2b_correlation_1H.png"))